# Video → 3D Reconstruction (VGGT-Omega + SAM 3)



## 1. Clone the repo and install Python dependencies
We use `requirements-colab.txt` because Colab already provides matched Torch / CUDA wheels — reinstalling Torch wastes ~3 minutes and sometimes breaks the runtime.

In [1]:
!git clone https://github.com/ayushmaankaria/Video-to-3D-Reconstruction.git
%cd Video-to-3D-Reconstruction

!pip install -q -r requirements-colab.txt

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Cloning into 'Video-to-3D-Reconstruction'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 159 (delta 73), reused 127 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 1.81 MiB | 34.37 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/Video-to-3D-Reconstruction
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/5

## 2. Authenticate with Hugging Face
Both VGGT-Omega and SAM 3 are gated. Before running this cell:

1. Request access at https://huggingface.co/facebook/vggt-omega and https://huggingface.co/facebook/sam3 .
2. Create a *Read* token at https://huggingface.co/settings/tokens .
3. In Colab's left sidebar, open the 🔑 **Secrets** panel, add a secret named `HF_TOKEN`, paste your token, and enable notebook access.

The cell below reads that secret and logs in non-interactively.

In [2]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
assert token, "Add an HF_TOKEN secret in the Colab Secrets panel and re-run."
login(token=token.strip())
print("Logged in to Hugging Face.")

Logged in to Hugging Face.


## 3a. (Option A) Use a video from Google Drive
Mount Drive and point `VIDEO_PATH` at your phone video. Skip this cell and use 3b instead if you'd rather upload directly.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

# Change this to the exact location of your video in Drive.
VIDEO_PATH = "/content/drive/MyDrive/desk_video.mp4"
!test -f "$VIDEO_PATH" && echo "Using $VIDEO_PATH" || echo "Video not found. Update VIDEO_PATH or use the upload fallback cell."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using /content/drive/MyDrive/desk_video.mp4


## 3b. (Option B) Upload a video directly
Pick a small video (under ~100 MB is fine). The selected file's path becomes `VIDEO_PATH`.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_name = next(iter(uploaded.keys()))
VIDEO_PATH = f"/content/Video-to-3D-Reconstruction/{desk_video.mp4}"
print("Using", VIDEO_PATH)

## 4. Download the VGGT-Omega checkpoint
Grabs the 512-resolution 1B checkpoint into `checkpoints/`. If you'd rather use the 256 + text-alignment variant, swap the repo id and pass `--image-resolution 256 --enable-alignment` in the next cell.

In [4]:
import os
from huggingface_hub import hf_hub_download

CORRECT_REPO_ID = "facebook/VGGT-Omega"
CHECKPOINT_FILE_NAME = "vggt_omega_1b_512.pt"
LOCAL_CKPT_DIR = "checkpoints/vggt-omega-1b-512"
FINAL_MODEL_PATH = os.path.join(LOCAL_CKPT_DIR, "model.pt")

# Ensure the local directory exists
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

# Download the specific checkpoint file
try:
    downloaded_file_path = hf_hub_download(
        repo_id=CORRECT_REPO_ID,
        filename=CHECKPOINT_FILE_NAME,
        local_dir=LOCAL_CKPT_DIR,
        # local_dir_use_symlinks=False # This argument is deprecated and ignored
    )
    print(f"File downloaded to: {downloaded_file_path}")

    # Rename the downloaded file to 'model.pt' as expected by the next cell
    if os.path.exists(downloaded_file_path):
        os.rename(downloaded_file_path, FINAL_MODEL_PATH)
        print(f"Checkpoint renamed to: {FINAL_MODEL_PATH}")
    else:
        print(f"Error: Downloaded file not found at {downloaded_file_path}")

except Exception as e:
    print(f"Error downloading checkpoint: {e}")
    downloaded_file_path = None # Indicate failure

print("Checkpoint directory:", LOCAL_CKPT_DIR)

vggt_omega_1b_512.pt:   0%|          | 0.00/4.58G [00:00<?, ?B/s]

File downloaded to: checkpoints/vggt-omega-1b-512/vggt_omega_1b_512.pt
Checkpoint renamed to: checkpoints/vggt-omega-1b-512/model.pt
Checkpoint directory: checkpoints/vggt-omega-1b-512


## 5. Run the reconstruction pipeline
This single CLI call does:
1. Sample frames from the video (`--max-frames`, hybrid sharp+uniform mode).
2. Run VGGT-Omega for dense depth + camera intrinsics/extrinsics.
3. Run SAM 3 once per frame for each text concept in `--concepts` and stamp out a label map.
4. Fuse everything into a colored, semantically-labeled point cloud and write `runs/desk/exports/`.

Tweak `--concepts` to match what's actually in your scene — extra concepts cost roughly +0.3 s/frame each.

In [7]:
!python -m spatial_recon.cli run \
  --video "$VIDEO_PATH" \
  --out runs/desk \
  --checkpoint checkpoints/vggt-omega-1b-512/model.pt \
  --image-resolution 512 \
  --max-frames 48 \
  --fps 2.0 \
  --concepts "desk,chair,monitor,keyboard,mouse,cup,wall,floor,lamp,cable,guitar,toothbrush,medicine,watch" \
  --conf-percentile 30 \
  --sample-stride 2 \
  --voxel-size 0.01

Extracting frames: 100% 48/48 [00:43<00:00,  1.11it/s]
[Video] Wrote 48 frames @ 1280x720 (hybrid) to runs/desk/frames
[Predict] Device: cuda
[Predict] GPU: NVIDIA L4
[Predict] Loading VGGT-Omega checkpoint: checkpoints/vggt-omega-1b-512/model.pt
[Predict] Preprocessed image tensor shape: (48, 3, 384, 688)
[Predict] Running VGGT-Omega…
[Predict] Forward pass complete.
[Predict] Wrote runs/desk/predictions.npz
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Semantics] SAM 3 on cuda; 14 concepts
config.json: 25.8kB [00:00, 8.25MB/s]
sam3.pt: 100% 3.45G/3.45G [00:11<00:00, 292MB/s]
SAM 3 segmentation: 100% 48/48 [01:04<00:00,  1.34s/it]
Fusing frames: 100% 48/48 [00:00<00:00, 102.23it/s]
[Pipeline] Fused 71,629 points across 48 frames
[Pipeline] Done. Open runs/des

## 6. Preview the interactive viewer inline
`viewer.html` is a self-contained Plotly scene — colored points, semantic toggle, and the camera trajectory. Rendering inside Colab works for clouds up to ~500k points; for bigger ones, download and open locally.

In [8]:
from IPython.display import HTML, display
display(HTML(filename="runs/desk/exports/viewer.html"))

## 7. Open-vocabulary 3D query (SAM 3 backed)
Re-runs SAM 3 with your text prompt over the original frames, then highlights the top-K% of fused points whose source pixels fall inside the highest-scoring SAM 3 masks. Output is `runs/desk/exports/query_<text>.ply`.

In [9]:
!python -m spatial_recon.cli query \
  --run runs/desk \
  --text "chair" \
  --topk-percent 8

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
SAM 3 query 'chair': 100% 48/48 [00:12<00:00,  3.91it/s]
[Query] SAM 3 produced 3,800,779 matching pixels across 48 frames
[Query] Selected 5,730 / 71,629 points (8.0%)
[Query] Wrote runs/desk/exports/query_chair.ply


## 8. Zip and download all outputs
Bundles frames, the VGGT predictions NPZ, SAM 3 label maps, fused PLY/GLB, the viewer HTML, the legend, the query PLY, and `REPORT.md`.

In [10]:
from google.colab import files
!zip -r desk_reconstruction_outputs.zip runs/desk
files.download("desk_reconstruction_outputs.zip")

  adding: runs/desk/ (stored 0%)
  adding: runs/desk/exports/ (stored 0%)
  adding: runs/desk/exports/reconstruction_semantic.glb (deflated 31%)
  adding: runs/desk/exports/viewer.html (deflated 71%)
  adding: runs/desk/exports/reconstruction_rgb.ply (deflated 10%)
  adding: runs/desk/exports/query_chair.ply (deflated 24%)
  adding: runs/desk/exports/fused_points.npz (deflated 0%)
  adding: runs/desk/exports/reconstruction_semantic.ply (deflated 24%)
  adding: runs/desk/exports/semantic_legend.json (deflated 77%)
  adding: runs/desk/frames_meta.json (deflated 34%)
  adding: runs/desk/frames/ (stored 0%)
  adding: runs/desk/frames/0035.jpg (deflated 0%)
  adding: runs/desk/frames/0037.jpg (deflated 0%)
  adding: runs/desk/frames/0000.jpg (deflated 0%)
  adding: runs/desk/frames/0036.jpg (deflated 0%)
  adding: runs/desk/frames/0033.jpg (deflated 0%)
  adding: runs/desk/frames/0028.jpg (deflated 0%)
  adding: runs/desk/frames/0009.jpg (deflated 0%)
  adding: runs/desk/frames/0014.jpg (de

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>